# 용역 클러스터링

대상: 용역(servc) 7,513건 제목 벡터 (중복 제거본, BGE-M3 1024차원, float16 저장)

## 왜 용역만 따로 하는가

공사(cnstwk)는 클러스터링이 실패했다 - 제목이 "(가칭)탕정9초등학교 교사 신축 전기공사"처럼 대부분 고유명사라, 공사 성격이 아니라 지역명(탕정/에코/아라)으로 그룹이 갈렸다. 게다가 공사는 `main_cnstty_nm` 컬럼이 97.5% 채워져 있어 클러스터링이 필요 없었다.

용역은 상황이 다르다:

- 제목에 **업무 성격이 드러난다** ("2026년 정보시스템 통합 유지관리 용역", "장애인실태조사")
- 조달청 코드(`item_codes`)가 **16.3%밖에 없어** 대안이 부족하다
- 초기 탐색에서 K=8로 돌렸을 때 해석 가능한 그룹이 나왔다 (연구용역/IT구축/실태조사/건설감리/유지관리/정보보호/운영위탁/기능개선)

## 목표

LLM 프롬프트에 넣을 **카테고리 목록**을 만드는 것. 클러스터 자체를 태그로 쓰는 게 아니라, 클러스터가 알려주는 "축"을 참고해 사람이 목록을 확정한다.

## 0. 환경 확인

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import umap

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("sklearn", sklearn.__version__)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

## 1. 데이터 로드

In [ ]:
from pathlib import Path

CACHE = Path("cache")

# float16으로 저장돼 있다(전송 크기 때문). sklearn 일부 함수가 float16에서
# 정밀도 경고를 내므로 float32로 올려서 쓴다 - 값 자체는 그대로다.
X = np.load(CACHE / "vectors_servc.npy").astype(np.float32)
df = pd.read_csv(CACHE / "meta_servc.csv")

print("벡터:", X.shape)
print("메타:", df.shape)
assert len(X) == len(df), "벡터와 메타 개수가 다르면 짝이 어긋난 것"
df.head()

## 2. 데이터 감 잡기

클러스터링 전에 어떤 제목들인지 눈으로 훑는다. 나중에 그룹 결과를 볼 때 기준이 된다.

In [ ]:
rng = np.random.default_rng(42)

df["title_len"] = df["title"].str.len()
print(df["title_len"].describe().round(1))

print("\n무작위 20건:")
for t in df["title"].sample(20, random_state=42):
    print(f"  {t[:70]}")

## 3. 그리드 탐색

차원축소와 클러스터링 파라미터는 서로 영향을 주므로 분리하지 않고 한 번에 탐색한다.
UMAP이 비싼 연산이라 바깥 루프에 두고, K는 안쪽에서 돌린다 - UMAP 1회 비용으로 K 여러 개를 본다.

**평가를 원본 1024차원에서 하는 이유**: UMAP은 이웃을 뭉치게 만드는 알고리즘이라 UMAP 공간에서 실루엣을 재면 "심하게 뭉치는 설정"이 무조건 이긴다. 원본 공간에서 재야 모든 UMAP 설정이 같은 잣대로 비교된다.

`min_dist=0.0` 고정 - 클러스터링 목적일 때의 표준값이다(0.1~0.5는 시각화용).

In [ ]:
import time
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

Xn = normalize(X)  # BGE-M3는 이미 정규화돼 있지만, 모델이 바뀌어도 안전하게

N_COMPONENTS = [5, 10, 20, 50]
N_NEIGHBORS = [5, 15, 50]
K_GRID = list(range(6, 21))

rows = []
t0 = time.perf_counter()
done = 0
total = len(N_COMPONENTS) * len(N_NEIGHBORS)

for n_comp in N_COMPONENTS:
    for n_nb in N_NEIGHBORS:
        emb = umap.UMAP(n_components=n_comp, n_neighbors=n_nb, min_dist=0.0,
                        metric="cosine", random_state=42).fit_transform(Xn)
        for k in K_GRID:
            labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(emb)
            sil = silhouette_score(Xn, labels, metric="cosine",
                                   sample_size=2000, random_state=42)
            rows.append({"n_components": n_comp, "n_neighbors": n_nb,
                         "K": k, "silhouette": sil})
        done += 1
        print(f"  UMAP {done}/{total} (n_comp={n_comp}, n_nb={n_nb}) "
              f"누적 {time.perf_counter()-t0:.0f}초", flush=True)

results = pd.DataFrame(rows)
results.to_csv(CACHE / "grid_servc.csv", index=False, encoding="utf-8-sig")
print(f"\n조합 {len(results)}개 완료")

### 3-1. 상위 조합

In [ ]:
results.sort_values("silhouette", ascending=False).head(15)

### 3-2. 파라미터별 경향

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for n_comp, g in results.groupby("n_components"):
    axes[0].plot(g.groupby("K")["silhouette"].max(), marker="o", label=f"n_comp={n_comp}")
axes[0].set_xlabel("K"); axes[0].set_ylabel("최고 실루엣")
axes[0].set_title("K별 (n_components 비교)"); axes[0].legend(); axes[0].grid(alpha=0.3)

for n_nb, g in results.groupby("n_neighbors"):
    axes[1].plot(g.groupby("K")["silhouette"].max(), marker="o", label=f"n_nb={n_nb}")
axes[1].set_xlabel("K"); axes[1].set_title("K별 (n_neighbors 비교)")
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(results.groupby("K")["silhouette"].max(), marker="o", color="black")
axes[2].set_xlabel("K"); axes[2].set_title("K별 전체 최고")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 그룹 내용 확인

**여기가 실제 판단 지점이다.** 실루엣 점수는 참고일 뿐이고, 그룹이 카테고리로 말이 되는지는 제목을 직접 읽어야 안다.

공사 클러스터링에서 중심 근처 표본만 보고 "잘 나뉘었다"고 판단했다가, 실제로는 지역명으로 갈린 걸 놓칠 뻔했다. 중심에 가까운 건 그 그룹의 가장 전형적인 것들이라 당연히 일관돼 보인다. 그래서 여기서는 **세 가지 표본을 함께** 본다:

- **중심 근처** - 그룹의 전형
- **무작위** - 편향 없는 실제 분포
- **경계(중심에서 먼 것)** - 이 그룹이 진짜 하나의 카테고리인지 판정하는 핵심

In [ ]:
def inspect(n_comp, n_nb, k, n_show=5):
    """지정한 조합으로 다시 클러스터링하고 그룹별 표본 3종을 출력한다.

    random_state가 고정돼 있어 그리드 탐색 때와 같은 결과가 재현된다.
    """
    emb = umap.UMAP(n_components=n_comp, n_neighbors=n_nb, min_dist=0.0,
                    metric="cosine", random_state=42).fit_transform(Xn)
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(emb)

    for c in range(k):
        idx = np.where(labels == c)[0]
        center = emb[idx].mean(axis=0)
        dist = np.linalg.norm(emb[idx] - center, axis=1)
        order = np.argsort(dist)

        print(f"\n{'='*66}")
        print(f"그룹 {c} - {len(idx)}건")
        print("-- 중심 근처(전형) --")
        for i in idx[order][:n_show]:
            print(f"   {df['title'][i][:64]}")
        print("-- 무작위 --")
        for i in rng.choice(idx, min(n_show, len(idx)), replace=False):
            print(f"   {df['title'][i][:64]}")
        print("-- 경계(중심에서 먼 것) --")
        for i in idx[order][-n_show:]:
            print(f"   {df['title'][i][:64]}")

    return labels

### 4-1. 최고 조합 확인

3-1의 상위 조합 값을 넣어서 실행한다.

In [ ]:
best = results.sort_values("silhouette", ascending=False).iloc[0]
print(f"최고 조합: n_comp={int(best.n_components)}, n_nb={int(best.n_neighbors)}, "
      f"K={int(best.K)}, 실루엣={best.silhouette:.4f}\n")

labels_best = inspect(int(best.n_components), int(best.n_neighbors), int(best.K))

### 4-2. K를 바꿔서 비교

실루엣이 최고인 K가 카테고리로도 최적이라는 보장은 없다. 더 굵게(K 작게) 또는 더 잘게(K 크게) 나눴을 때 어느 쪽이 이름 붙이기 좋은지 직접 비교한다.

In [ ]:
# 원하는 조합으로 바꿔가며 실행
labels_alt = inspect(n_comp=10, n_nb=15, k=10)

## 5. 결론

실행 후 채운다.

- 채택한 조합 (n_components / n_neighbors / K / 실루엣):
- 그룹이 카테고리로 말이 되는가 (경계 표본까지 봤을 때):
- 공사처럼 고유명사로 갈린 그룹이 있는가:
- **카테고리 목록 초안**:
    1. 
    2. 
    3. 
- 클러스터링으로 안 잡히는 성격이 있는가 (코드 그룹과 대조 필요):

### 다음 단계

- 데스크탑에서 조달청 코드(`item_codes`) 그룹과 대조 - 클러스터와 코드가 얼마나 일치하는지(ARI)
- 목록 확정 후 `pipeline/realtime/src/extractors/llm/` 프롬프트에 반영